# The Voice Agent Loop [Agent Patterns - Module 11]

> **MLCourse - Agentic AI - Agent Patterns**

Now we close the loop: audio in, agent, audio out. Three stages, each an
independent, testable component.

```
caller.wav --[whisper-large-v3]--> transcript --[qwen/qwen3.8-27b]--> reply --[TTS]--> reply.wav
```

The engineering point of this notebook is the **seam discipline**: keep the
three stages separate, log the text at each boundary, and you can debug a
voice agent the same way you debug a text one. Blur them and you get bug
reports like "it misunderstood me" with no way to tell which stage failed.

### What you will learn

1. Wiring STT -> agent -> TTS as three independent functions.
2. Writing prompts for speech output (which differ from prompts for screens).
3. The text-to-speech step, including what Groq's TTS currently requires.
4. A measured latency budget across the three stages.

### Key takeaways

- Log the transcript and the reply text. They are your entire debug trail.
- A reply written for a screen sounds terrible read aloud. Prompt for the ear.
- Latency is the product. Measure each stage separately.

### A note on text-to-speech in this notebook

Groq hosts a TTS model, `canopylabs/orpheus-v1-english`. **At the time this
notebook was written it returns HTTP 400 `model_terms_required` on this
account** - Groq requires an organisation admin to accept the model's terms
once, in the Groq console, before any request succeeds.

Rather than hide that, the notebook **calls the Groq endpoint every run and
prints the real response**. For the audio itself it then uses `pyttsx3`,
which drives the operating system's built-in voice - offline, keyless, and
good enough to prove the loop closes. If your organisation has accepted the
terms, the Groq branch will simply take over and write the file.

### Setup: imports, environment, track discovery


In [ ]:
import os
import re
import sys
import json
import time
import wave
import random
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives INSIDE the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]   # loud failure if missing, by design
CHAT_MODEL = "qwen/qwen3.8-27b"              # Groq-hosted; never OpenAI
STT_MODEL = "whisper-large-v3"               # Groq-hosted speech-to-text

HERE = Path.cwd().resolve()
AUDIO = HERE / "audio"
AUDIO.mkdir(exist_ok=True)

print(f"Track root : {TRACK}")
print(f"Audio dir  : {AUDIO}")
print(f"Chat model : {CHAT_MODEL}")
print(f"STT model  : {STT_MODEL}")


### Making audio without a microphone


In [ ]:
# This module must run headless, so there is no mic. We SYNTHESISE the input
# audio with pyttsx3, which drives the operating system's built-in TTS voice
# (SAPI5 on Windows, NSSpeechSynthesizer on macOS, espeak on Linux).
#
# It is fully offline, needs no key, and gives us a real .wav file with real
# speech in it - which is exactly what the STT step needs.

import pyttsx3

def speak_to_file(text, path, rate=150):
    """Render `text` to a .wav file using the OS voice. Returns the Path."""
    path = Path(path)
    engine = pyttsx3.init()
    engine.setProperty("rate", rate)          # words per minute
    engine.save_to_file(text, str(path))
    engine.runAndWait()
    engine.stop()
    return path

def wav_info(path):
    with wave.open(str(path)) as w:
        return {
            "seconds": round(w.getnframes() / w.getframerate(), 2),
            "sample_rate": w.getframerate(),
            "channels": w.getnchannels(),
            "bytes": Path(path).stat().st_size,
        }

print("speak_to_file() ready (offline OS voice)")


### Groq speech-to-text


In [ ]:
# whisper-large-v3 on Groq. Multipart upload: the file goes in `files`, the
# parameters go in `data`. Backoff included - the free tier is shared.

import requests

STT_URL = "https://api.groq.com/openai/v1/audio/transcriptions"

def transcribe(path, response_format="json", language="en"):
    """Send a .wav to Groq whisper-large-v3 and return the parsed response."""
    for attempt in range(5):
        with open(path, "rb") as fh:
            r = requests.post(
                STT_URL,
                headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
                files={"file": (Path(path).name, fh, "audio/wav")},
                data={"model": STT_MODEL,
                      "response_format": response_format,
                      "language": language,
                      "temperature": 0},
                timeout=120,
            )
        if r.status_code == 200:
            return r.json()
        wait = 2 ** attempt + random.random()
        print(f"  HTTP {r.status_code}, retry {attempt+1} in {wait:.1f}s")
        time.sleep(wait)
    raise RuntimeError(f"STT failed: {r.status_code} {r.text[:300]}")

print("transcribe() ready")


### Groq chat, with 429 backoff


In [ ]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

def ask(prompt, system="You are a precise assistant.", max_tokens=500, temperature=0.0):
    for attempt in range(5):
        try:
            r = client.chat.completions.create(
                model=CHAT_MODEL,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": prompt}],
                temperature=temperature, max_tokens=max_tokens,
            )
            return r.choices[0].message.content
        except Exception as e:
            wait = 2 ** attempt + random.random()
            print(f"  retry {attempt+1} in {wait:.1f}s ({type(e).__name__})")
            time.sleep(wait)
    raise RuntimeError("Groq chat failed after 5 attempts")

print("ask() ready")


### 1. Stage one: the caller speaks


### Generate and transcribe the caller audio


In [ ]:
SCRIPT = ("Hello, my dishwasher is leaking from the bottom and there is water "
          "on the kitchen floor. I bought it about two years ago. "
          "What should I do right now?")

call_wav = speak_to_file(SCRIPT, AUDIO / "support_call.wav")

t0 = time.time()
transcript = transcribe(call_wav)["text"].strip()
stt_ms = (time.time() - t0) * 1000

print("audio    :", call_wav.name, wav_info(call_wav))
print("transcript:", transcript)
print(f"STT      : {stt_ms:.0f} ms")


### 2. Stage two: the agent thinks

This is an ordinary Groq chat call. The only thing that changes is the
**system prompt**, and it changes a lot.

Text written for a screen and text written for the ear are different
artifacts. On a screen, a bulleted list is helpful. Read aloud, "bullet one,
bullet two" is unbearable. Rules that actually matter:

- **short** - 2-4 sentences; a listener cannot skim,
- **no markdown** - asterisks and hashes get read out or mangled,
- **no lists** - use "first... then..." instead,
- **front-load the answer** - the listener cannot scroll back,
- **spell out awkward tokens** - "R M A dash eight eight one two", not
  "RMA-8812", unless your TTS handles it.

### The agent


In [ ]:
VOICE_SYSTEM = (
    "You are a calm home-appliance support agent speaking to a caller ON THE "
    "PHONE. Your reply will be read aloud by a speech synthesiser.\n"
    "Rules:\n"
    "- At most 3 short sentences.\n"
    "- Plain spoken English. No markdown, no bullet points, no numbered lists.\n"
    "- Give the single most important safety action first.\n"
    "- End with one short question to move the call forward."
)

t0 = time.time()
reply = ask(transcript, system=VOICE_SYSTEM, max_tokens=250).strip()
llm_ms = (time.time() - t0) * 1000

print("reply:")
print(" ", reply)
print(f"\nLLM: {llm_ms:.0f} ms")


### Compare against a screen-style reply


In [ ]:
# Same question, default prompt. Read both aloud in your head.

screen_reply = ask(transcript,
                   system="You are a helpful home-appliance support agent.",
                   max_tokens=400).strip()

print("=== screen-style reply ===")
print(screen_reply[:700])
print()
print(f"voice reply : {len(reply):>4} chars, {reply.count(chr(10))+1} line(s)")
print(f"screen reply: {len(screen_reply):>4} chars, {screen_reply.count(chr(10))+1} line(s)")
print()
print("At ~150 words per minute, the screen reply takes far longer to hear")
print("than to read - and any markdown in it becomes noise in the audio.")


### 3. Stage three: the agent speaks

First we try Groq. Whatever it answers is printed verbatim.

### Speaking back: Groq TTS first, OS voice as the working fallback


In [ ]:
# We TRY the Groq TTS endpoint every run and print the real response, so the
# notebook never hides what the API actually said.

import requests

TTS_URL = "https://api.groq.com/openai/v1/audio/speech"
GROQ_TTS_MODEL = "canopylabs/orpheus-v1-english"

def groq_tts(text, path, voice="tara"):
    """Attempt Groq TTS. Returns (ok, status_code, message)."""
    r = requests.post(
        TTS_URL,
        headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
        json={"model": GROQ_TTS_MODEL, "input": text,
              "voice": voice, "response_format": "wav"},
        timeout=120,
    )
    if r.status_code == 200:
        Path(path).write_bytes(r.content)
        return True, 200, f"wrote {len(r.content)} bytes"
    return False, r.status_code, r.text[:300]

print("groq_tts() ready")


### Attempt Groq TTS


In [ ]:
groq_path = AUDIO / "reply_groq.wav"
ok, status, detail = groq_tts(reply, groq_path)

print(f"Groq TTS ({GROQ_TTS_MODEL})")
print(f"  status : {status}")
print(f"  detail : {detail}")
print()
if ok:
    print("Groq TTS succeeded - the terms have been accepted for this org.")
else:
    print("Groq TTS unavailable on this account.")
    print("If the status is 400 with 'model_terms_required', an org admin must")
    print("accept the model terms once at console.groq.com; nothing in the code")
    print("can work around it. Falling back to the offline OS voice below.")


### Produce the audio


In [ ]:
reply_path = AUDIO / "reply.wav"

t0 = time.time()
if ok:
    reply_path = groq_path
    engine_used = f"groq/{GROQ_TTS_MODEL}"
else:
    speak_to_file(reply, reply_path)
    engine_used = "pyttsx3 (offline OS voice)"
tts_ms = (time.time() - t0) * 1000

print("engine :", engine_used)
print("file   :", reply_path.name)
print("info   :", wav_info(reply_path))
print(f"TTS    : {tts_ms:.0f} ms")
assert reply_path.exists() and reply_path.stat().st_size > 1000, "no audio produced"
print("\nReal audio file written. Play it with any media player.")


### Listen to it in the notebook


In [ ]:
# IPython renders an inline player. The audio is embedded in the saved
# notebook, so it survives a re-open.

from IPython.display import Audio, display

print("caller said:")
display(Audio(str(call_wav)))
print("agent replied:")
display(Audio(str(reply_path)))


### 4. The latency budget

Voice is a real-time medium. On a phone call, a pause over about **800 ms**
reads as "did it hang up?" That budget has to cover all three stages plus
the network.

Measure them separately - the fix for a slow STT stage (smaller model,
streaming) is nothing like the fix for a slow LLM stage (shorter output,
fewer tokens).

### Where the time went


In [ ]:
total = stt_ms + llm_ms + tts_ms
audio_s = wav_info(call_wav)["seconds"]

print(f"{'stage':10s}{'ms':>9s}{'share':>9s}")
print("-" * 28)
for name, ms in [("STT", stt_ms), ("agent", llm_ms), ("TTS", tts_ms)]:
    print(f"{name:10s}{ms:>9.0f}{ms/total:>8.0%}")
print("-" * 28)
print(f"{'TOTAL':10s}{total:>9.0f}")
print()
print(f"caller audio was {audio_s:.1f}s long")
print()
print("Note what this measurement does NOT include: waiting for the caller to")
print("stop talking. In a live system, endpointing (deciding the turn is over)")
print("often costs more than any single stage here.")


### Reading the budget

The usual shape is that the **agent** stage dominates, because it is the only
one that generates tokens. Which gives you the cheapest optimisation in
voice: **make the reply shorter**. It cuts LLM time and TTS time at once, and
it also makes the reply better to listen to. The prompt in section 2 was
already doing double duty.

Beyond that: stream the LLM output into the TTS sentence by sentence so the
caller hears the first sentence while the rest is still generating. That is
how production voice agents hide the latency, and it is why the three stages
must stay separable.

### Pitfalls recap

- **One giant function.** Keep STT, agent and TTS separate or you cannot tell
  which one failed.
- **Screen prompts for voice.** Markdown, lists and long paragraphs are all
  wrong for the ear.
- **Not logging the transcript.** When a caller complains the agent
  misunderstood, the transcript is the evidence.
- **Assuming a TTS provider is available.** Model terms, region limits and
  quota all bite. Have a working fallback and say plainly which one ran.
- **Optimising the wrong stage.** Measure before you tune.

### Next

Notebook 03 handles the fact that the transcript is sometimes wrong.